In [ ]:
# Author: M. Riley Owens (GitHub: mrileyowens)

# This file plots the SFHs of the BEAGLE 2CSFH fits

In [ ]:
import sys

import os
import glob

import numpy as np

from astropy.io import fits

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile

In [ ]:
def plot():

    '''
    Plot the SFHs of the 2CSFH BEAGLE models
    '''

    # Set common directories
    home = os.getcwd()
    data = f'{home}/data'
    figs = f'{home}/figs'
    results = f'{home}/results'

    # Get the corresponding files of the BEAGLE fits
    files = glob.glob(f'{results}/beagle_fits/e24_f775w_dropouts_2csfh_no_lya/*_GOODS*_BEAGLE.fits.gz')

    # For each BEAGLE fit results file
    for i, file in enumerate(files):

        # Check if the file is empty; skip if so
        if os.stat(file).st_size == 0:
            print(f'File {os.path.basename(file)} is empty. Skipping.')
            continue

        # Get the galaxy ID from the file name
        id = os.path.basename(file).split('_BEAGLE')[0]

        # Get the HDUL of the BEAGLE fit results file
        hdul = fits.open(file)

        # Get the probabilities of each entry in the posterior
        probs = hdul['POSTERIOR PDF'].data['probability']

        # Get the 2CSFH model parameters necessary to calculate the SFH
        m_tot = hdul['GALAXY PROPERTIES'].data['M_tot']
        cst = 10**hdul['POSTERIOR PDF'].data['current_sfr_timescale']
        msa = 10**hdul['POSTERIOR PDF'].data['max_stellar_age']
        sfr = hdul['STAR FORMATION'].data['SFR']
        tau = 10**hdul['POSTERIOR PDF'].data['tau']

        # Make an empty array of the normalizing constants for the SFH
        alpha = np.zeros(np.shape(m_tot), dtype=np.float64)

        # For each entry in the posterior
        for j, _ in enumerate(cst):

            # Make a time array spanning the duration of the exponential decay portion of the SFH
            t = np.arange(cst[j], msa[j] + 1e5, 1e5)

            # Calculate the normalizing constant, assuming that the integral of the two SFH components matches the total stellar mass formed
            alpha[j] = (m_tot[j] - sfr[j] * cst[j]) / np.trapezoid(t * np.exp(-t / tau[j]), t)

        # Make an array of lookback times between the current time and the maximum stellar age
        t = np.arange(0, np.max(msa) + 1e5, 1e5)
        
        # Make an empty array of the SFHs
        sfhs = np.zeros((len(m_tot), len(t)))

        # For each SFH
        for j, _ in enumerate(sfhs):

            # Assemble the two components of the SFH
            #sfhs[j] = np.where(t <= cst[j], sfr[j], alpha[j] * t * np.exp(-t / tau[j]))
            #sfhs[j] = np.where(t > msa[j], 0, sfhs[j])

            sfhs[j] = np.where(t <= cst[j], sfr[j], alpha[j] * (msa[j] - t) * np.exp(-(msa[j] - t) / tau[j]))
            sfhs[j] = np.where(t > msa[j], 0, sfhs[j])

        # Make a new figure of the SFH
        fig, ax = plt.subplots()

        # Make empty arrays to contain properties of the SFH posterior distribution at each lookback time
        medians, lowers, uppers = np.zeros(len(t)), np.zeros(len(t)), np.zeros(len(t))

        # For each lookback time
        for j, _ in enumerate(t):

            # Calculate the median and 16th and 84th percentiles of the SFH
            median, lower, upper = weighted_quantile(sfhs[:,j], probs, [0.5, 0.16, 0.84])

            # Add the percentiles to the complete array
            medians[j] = median
            lowers[j] = lower
            uppers[j] = upper

        # Plot the median SFH with the 16th and 84th percentile envelope shaded
        ax.plot(t / 1e6, medians, c='black')
        ax.fill_between(t / 1e6, lowers, uppers, color='black', alpha=0.2)

        # Label the figure
        ax.set_xlabel('Lookback time (Myr)')
        ax.set_ylabel('Star formation rate (M$_\odot$ yr$^{-1}$)')

        # Set the coordinate limits of the figure
        ax.set_xlim(left=1)
        ax.set_ylim(bottom=0)

        # Scale the lookback time logarithmically
        ax.set_xscale('log')

        # Label the figure with the object's ID
        at = AnchoredText(id, loc='upper right', frameon=False, prop=dict(fontweight='bold'))
        ax.add_artist(at)

        # Save the figure
        fig.savefig(f'{figs}/sfhs/sfh_{id}.png', bbox_inches='tight', dpi=200)

        plt.close('all')

In [ ]:
hdul = fits.open(f'data/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')
print(hdul[1].data)

In [ ]:
plot()